In [1]:
file_path="../data/"

In [2]:
import pandas as pd
import numpy as np

In [3]:
orders = pd.read_csv(file_path + 'orders.csv')
products = pd.read_csv(file_path + 'products.csv')
returns = pd.read_csv(file_path + 'returns.csv')
web_traffic = pd.read_csv(file_path + 'web_traffic.csv')
order_items = pd.read_csv(file_path + 'order_items.csv')
customers = pd.read_csv(file_path + 'customers.csv')
geography = pd.read_csv(file_path + 'geography.csv')
payments = pd.read_csv(file_path + 'payments.csv')

C:\Users\ASUS\AppData\Local\Temp\ipykernel_5420\519834596.py:5: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(file_path + 'order_items.csv')


### Q1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu?

**Dữ liệu:** `orders.csv`

In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['order_date'].isna().sum()

np.int64(0)

In [5]:
q1_orders = orders.sort_values(by=['customer_id', 'order_date'])
q1_orders['prev_order_date'] = q1_orders.groupby('customer_id')['order_date'].shift(1) # cách xuống 1 dòng
q1_orders['inter_order_gap'] = (q1_orders['order_date'] - q1_orders['prev_order_date']).dt.days
q1_ans = q1_orders['inter_order_gap'].median()
print(f"Q1 Answer: {q1_ans} days")

Q1 Answer: 144.0 days


### Q2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất?

**Công thức:** `(price - cogs) / price`

In [15]:
products['margin'] = (products['price'] - products['cogs']) / products['price']
q2_ans = products.groupby('segment')['margin'].mean().idxmax() # Tính margin trung bình theo từng segment đồng thời lấy max
print(f"Q2 Answer: {q2_ans}")

Q2 Answer: Standard


### Q3: Lý do trả hàng nào xuất hiện nhiều nhất cho danh mục 'Streetwear'?

In [16]:
q3_merged = returns.merge(products, on='product_id')  # Gộp bảng returns với products theo product_id
q3_streetwear = q3_merged[q3_merged['category'] == 'Streetwear']
q3_ans = q3_streetwear['return_reason'].value_counts().idxmax() # Đếm số lần xuất hiện của từng lý do trả hàng đồng thời lấy max
print(f"Q3 Answer: {q3_ans}")

Q3 Answer: wrong_size


### Q4: Nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất?

In [8]:
q4_ans = web_traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin()
print(f"Q4 Answer: {q4_ans}")

Q4 Answer: email_campaign


### Q5: Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi?

In [9]:
total_items = len(order_items)
promo_items = order_items['promo_id'].notna().sum()
q5_ans = (promo_items / total_items) * 100
print(f"Q5 Answer: {q5_ans:.2f}%")

Q5 Answer: 38.66%


### Q6: Nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất?

**Công thức:** `tổng số đơn / số khách hàng trong nhóm`

In [10]:
q6_merged = orders.merge(customers.dropna(subset=['age_group']), on='customer_id')
q6_stats = q6_merged.groupby('age_group').agg(
    total_orders=('order_id', 'nunique'),
    total_customers=('customer_id', 'nunique')
)
q6_stats['avg_orders'] = q6_stats['total_orders'] / q6_stats['total_customers']
q6_ans = q6_stats['avg_orders'].idxmax()
print(f"Q6 Answer: {q6_ans}")

Q6 Answer: 55+


### Q7: Vùng (region) nào tạo ra tổng doanh thu cao nhất?

**Công thức:** `Doanh thu = (Số lượng * Đơn giá) - Giảm giá`

In [11]:
q7_merged = order_items.merge(orders, on='order_id').merge(geography, on='zip')
# Điền 0 cho các dòng không có giảm giá (NaN)
q7_merged['discount_amount'] = q7_merged['discount_amount'].fillna(0)
q7_merged['revenue'] = (q7_merged['quantity'] * q7_merged['unit_price']) - q7_merged['discount_amount']
q7_ans = q7_merged.groupby('region')['revenue'].sum().idxmax()
print(f"Q7 Answer: {q7_ans}")

Q7 Answer: East


### Q8: Trong các đơn hàng bị hủy (`order_status = 'cancelled'`), phương thức thanh toán nào được sử dụng nhiều nhất?

In [12]:
q8_cancelled = orders[orders['order_status'] == 'cancelled']
q8_ans = q8_cancelled['payment_method'].value_counts().idxmax()
print(f"Q8 Answer: {q8_ans}")

Q8 Answer: credit_card


### Q9: Trong bốn kích thước (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất?

**Định nghĩa:** `Số bản ghi trả hàng / Số bản ghi mục đơn hàng`

In [13]:
q9_returns_merged = returns.merge(products, on='product_id')
returns_by_size = q9_returns_merged['size'].value_counts()

q9_orders_merged = order_items.merge(products, on='product_id')
orders_by_size = q9_orders_merged['size'].value_counts()


return_rates = returns_by_size / orders_by_size
# Lọc chỉ lấy 4 size được yêu cầu và tìm ra size có tỷ lệ cao nhất
target_sizes = ['S', 'M', 'L', 'XL']
filtered_rates = return_rates[return_rates.index.isin(target_sizes)]

q9_ans = filtered_rates.idxmax()
print(f"Q9 Answer: {q9_ans}")

Q9 Answer: S


### Q10: Kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

In [14]:
q10_ans = payments.groupby('installments')['payment_value'].mean().idxmax()
print(f"Q10 Answer: {q10_ans}")

Q10 Answer: 6
